In [ ]:
"""
Track the tiny loop of departure directions around W+'s unstable eigenspace
as it evolves under the flow, plotted in the (A, P_B) plane at several times.

Sigma = {A=0, P_B=0} is the origin of this plane. The loop starts far from
the origin (near (A0,0)) and should stay clear of it if W+ never connects
directly to Sigma.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.special import jv

# ---- parameters ----
# couplings derived from the pump
epsilon_p = .2
E_J=37 * 2 * np.pi
phi_a=0.11
phi_b=0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2
eps_d = 2 * alpha_sq * g2

delta = kappa_a*kappa_b/(4*g**2)

In [ ]:
A0 = np.sqrt(eps_d / g)

def rhs(t, y):
    A, B, PA, PB = y
    dA  = np.sqrt(2)*g*(PA*PB - A*B) - kappa_a/2*A - kappa_a*PA
    dB  = (g/np.sqrt(2))*(A**2+PA**2) - eps_d/np.sqrt(2) - kappa_b/2*B - kappa_b*PB
    dPA = np.sqrt(2)*g*(B*PA - A*PB) + kappa_a/2*PA
    dPB = np.sqrt(2)*g*A*PA + kappa_b/2*PB
    return [dA, dB, dPA, dPB]

def jacobian(y, eps=1e-6):
    n = len(y); J = np.zeros((n, n)); f0 = np.array(rhs(0, y))
    for i in range(n):
        yp = np.array(y, dtype=float); yp[i] += eps
        J[:, i] = (np.array(rhs(0, yp)) - f0) / eps
    return J

# ---- unstable eigenspace at W+ ----
Wplus = np.array([np.sqrt(A0**2-delta), 0, 0, 0])
w, v = np.linalg.eig(jacobian(Wplus))
idx_u = np.where(w.real > 1e-8)[0]
u1, u2 = v[:, idx_u[0]].real, v[:, idx_u[1]].real
u1 /= np.linalg.norm(u1)
u2 /= np.linalg.norm(u2)
idx_u = np.where(w.real > 0)[0][np.argsort(w[np.where(w.real > 0)[0]].real)]   # unstable, slow->fast
idx_s = np.where(w.real < 0)[0]
order = np.concatenate([idx_u, idx_s])
V = v[:, order].real                        # columns = [u1(slow), u2(fast), s1, s2]
Vinv = np.linalg.inv(V)

# ---- integrate the whole loop of directions ----
EPS = 1e-3
N_THETA = 1000
thetas = np.linspace(0, 2 * np.pi, N_THETA, endpoint=False)
CHECK_TIMES = np.linspace(1,6,50)

def ev_blow(t, y):
    return 1e5 - np.linalg.norm(y)
ev_blow.terminal = True
ev_blow.direction = -1

A_loop = np.zeros((len(CHECK_TIMES), N_THETA))
PB_loop = np.zeros((len(CHECK_TIMES), N_THETA))
Proj_u1_loop = np.zeros((len(CHECK_TIMES), N_THETA))
Proj_u2_loop = np.zeros((len(CHECK_TIMES), N_THETA))


for i, theta in enumerate(thetas):
    print(theta/2/np.pi)
    direction = np.cos(theta) * u1 + np.sin(theta) * u2
    y0 = Wplus + EPS * direction
    sol = solve_ivp(rhs, [0, max(CHECK_TIMES)], y0, max_step=0.02,
                     rtol=1e-10, atol=1e-12, events=[ev_blow], dense_output=True)
    t_end = sol.t[-1]
    for j, t_check in enumerate(CHECK_TIMES):
        Y = sol.sol(min(t_check, t_end))
        A_loop[j, i] = Y[0]
        PB_loop[j, i] = Y[3]
        coords = Vinv @ (Y - Wplus)
        Proj_u1_loop[j, i] = coords[0]
        Proj_u2_loop[j, i] = coords[1]

min_distance_to_Sigma = np.sqrt(A_loop**2 + PB_loop**2).min(axis=1)

In [ ]:
# ---- plot: one panel per checkpoint time ----
fig, axes = plt.subplots(6, len(CHECK_TIMES)//6, figsize=(16*2, 8*6))
for j, (ax, t_check) in enumerate(zip(axes.flat, CHECK_TIMES)):
    A_closed = np.append(A_loop[j], A_loop[j, 0])
    PB_closed = np.append(PB_loop[j], PB_loop[j, 0])
    ax.scatter(A_closed, PB_closed, c = range(N_THETA+1))
    ax.plot(0, 0, 'x', color='#993C1D', ms=10, mew=2)  # Sigma = origin
    ax.set_title(f"t = {t_check}")
    ax.set_xlabel("A")
    ax.set_ylabel("P_B")
    ax.set_xlim((-10,10))
    ax.set_ylim((-10,10))
    ax.ticklabel_format(style='sci', axis='both', scilimits=(-2, 2))

fig.suptitle("Loop of departure directions around W+, evolved forward, in the (A, P_B) plane\n"
             "(x marks the origin = Sigma; the loop never encloses it)")
plt.tight_layout()
#plt.savefig("loop_evolution.png", dpi=150)
plt.show()
#print("saved loop_evolution.png")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(CHECK_TIMES, min_distance_to_Sigma)
ax.set_ylabel("minimal distance to"+r'$\Sigma$')
ax.set_xlabel("Simulation time")
plt.show()

In [ ]:
# ---- plot: one panel per checkpoint time ----
fig, ax = plt.subplots(figsize=(6, 6))
cmap = plt.cm.viridis
for (j, t_check) in enumerate(CHECK_TIMES):
    Proj_u1_closed = np.append(Proj_u1_loop[j], Proj_u1_loop[j, 0])
    Proj_u2_closed = np.append(Proj_u2_loop[j], Proj_u2_loop[j, 0])
    ax.scatter(Proj_u1_closed, Proj_u2_closed, color=cmap(j/(len(CHECK_TIMES)-1)), lw=2, label=f"t={t_check}")

ax.set_xlabel("u_1")
ax.set_ylabel("u_2")
ax.set_yscale('symlog', linthresh=1e-3)
ax.set_xscale('symlog', linthresh=1e-3)
#ax.legend(fontsize=9)
ax.set_title("The loop on the unstable eigenspace (u1,u2)")
plt.tight_layout()
plt.show()


In [ ]:
kappa_b_list = [0.1]#, 0.4]
min_distance_to_Sigma_list = []
for kappa_b in kappa_b_list:
    A0 = np.sqrt(eps_d / g)

    def rhs(t, y):
        A, B, PA, PB = y
        dA  = np.sqrt(2)*g*(PA*PB - A*B)
        dB  = (g/np.sqrt(2))*(A**2+PA**2) - eps_d/np.sqrt(2) - kappa_b/2*B - kappa_b*PB
        dPA = np.sqrt(2)*g*(B*PA - A*PB)
        dPB = np.sqrt(2)*g*A*PA + kappa_b/2*PB
        return [dA, dB, dPA, dPB]

    def jacobian(y, eps=1e-6):
        n = len(y); J = np.zeros((n, n)); f0 = np.array(rhs(0, y))
        for i in range(n):
            yp = np.array(y, dtype=float); yp[i] += eps
            J[:, i] = (np.array(rhs(0, yp)) - f0) / eps
        return J

    # ---- unstable eigenspace at W+ ----
    w, v = np.linalg.eig(jacobian(Wplus))
    def real_pair(w, v, idx):
        """Two real vectors spanning the real invariant plane for this
        eigenvalue pair -- works whether it's two distinct reals or a
        complex-conjugate pair."""
        i0, i1 = idx
        if abs(w[i0].imag) > 1e-8:      # complex-conjugate pair
            return v[:, i0].real, v[:, i0].imag
        else:                            # real, distinct eigenvalues
            return v[:, i0].real, v[:, i1].real

    idx_u = np.where(w.real > 1e-8)[0]
    idx_u = idx_u[np.argsort(w[idx_u].real)]   # harmless tie in the complex case
    idx_s = np.where(w.real < -1e-8)[0]

    u1, u2 = real_pair(w, v, idx_u)
    s1, s2 = real_pair(w, v, idx_s)
    u1 /= np.linalg.norm(u1); u2 /= np.linalg.norm(u2)
    s1 /= np.linalg.norm(s1); s2 /= np.linalg.norm(s2)

    V = np.column_stack([u1, u2, s1, s2])      # columns = [u1, u2, s1, s2]
    try:
        Vinv = np.linalg.inv(V)
    except Exception as exc:
        print(exc)
        continue

    # ---- integrate the whole loop of directions ----
    EPS = 1e-3
    N_THETA = 100
    thetas = np.linspace(0, 2 * np.pi, N_THETA, endpoint=False)
    CHECK_TIMES = np.linspace(1,500,300)

    def ev_blow(t, y):
        return 1e5 - np.linalg.norm(y)
    ev_blow.terminal = True
    ev_blow.direction = -1

    A_loop = np.zeros((len(CHECK_TIMES), N_THETA))
    PB_loop = np.zeros((len(CHECK_TIMES), N_THETA))
    Proj_u1_loop = np.zeros((len(CHECK_TIMES), N_THETA))
    Proj_u2_loop = np.zeros((len(CHECK_TIMES), N_THETA))


    for i, theta in enumerate(thetas):
        print(theta/2/np.pi)
        direction = np.cos(theta) * u1 + np.sin(theta) * u2
        y0 = Wplus + EPS * direction
        sol = solve_ivp(rhs, [0, max(CHECK_TIMES)], y0, max_step=0.02,
                        rtol=1e-10, atol=1e-12, events=[ev_blow], dense_output=True)
        t_end = sol.t[-1]
        for j, t_check in enumerate(CHECK_TIMES):
            Y = sol.sol(min(t_check, t_end))
            A_loop[j, i] = Y[0]
            PB_loop[j, i] = Y[3]
            coords = Vinv @ (Y - Wplus)
            Proj_u1_loop[j, i] = coords[0]
            Proj_u2_loop[j, i] = coords[1]

    min_distance_to_Sigma = np.sqrt(A_loop**2 + PB_loop**2).min(axis=1)
    min_distance_to_Sigma_list.append(min_distance_to_Sigma)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(CHECK_TIMES, min_distance_to_Sigma_list[0])
ax.set_ylabel("minimal distance to"+r'$\Sigma$')
ax.set_xlabel("Simulation time")
ax.set_ylim(0,10)
plt.show()

In [ ]:
# ---- plot: one panel per checkpoint time ----
fig, ax = plt.subplots(figsize=(6, 6))
cmap = plt.cm.viridis
for (j, t_check) in enumerate(CHECK_TIMES):
    Proj_u1_closed = np.append(Proj_u1_loop[j], Proj_u1_loop[j, 0])
    Proj_u2_closed = np.append(Proj_u2_loop[j], Proj_u2_loop[j, 0])
    ax.scatter(Proj_u1_closed, Proj_u2_closed, color=cmap(j/(len(CHECK_TIMES)-1)), lw=2, label=f"t={t_check}")

ax.set_xlabel("u_1")
ax.set_ylabel("u_2")
ax.set_yscale('symlog', linthresh=1e-3)
ax.set_xscale('symlog', linthresh=1e-3)
#ax.legend(fontsize=9)
ax.set_title("The loop on the unstable eigenspace (u1,u2)")
plt.tight_layout()
plt.show()
